# Exploratory Data Analysis — Karachi AQI

Run this after the backfill has populated the feature store. Every figure here
either informs a feature-engineering decision or goes into the final report.

Questions this notebook answers:
1. How bad is Karachi's air, and how does that vary through the year?
2. Which meteorological variables actually drive it?
3. How persistent is the series — that is, how hard is persistence to beat?
4. Where are the data quality problems?
5. Does our EPA AQI implementation agree with the API's convenience field?

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import CITY, AQI_BANDS, band_for
from src.store import local_store

plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 10})

daily = local_store.read_daily()
hourly = local_store.read_hourly()
observed = daily[daily["aqi_mean"].notna()]
print(f"{CITY.name}: {len(observed):,} days, {observed.index.min().date()} to {observed.index.max().date()}")
print(f"hourly rows: {len(hourly):,}")
observed[["aqi_mean", "aqi_max", "pm2_5_mean", "wind_speed_10m_mean"]].describe().round(1)

## 1. How bad is it, and when?

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
for b in AQI_BANDS:
    ax.axhspan(b.lower, min(b.upper, 500), color=b.color, alpha=0.13)
ax.plot(observed.index, observed["aqi_mean"], lw=0.8, color="#1f2937", label="daily mean")
ax.plot(observed.index, observed["aqi_mean"].rolling(30).mean(), lw=2.5,
        color="#a21caf", label="30-day mean")
ax.set_ylabel("US AQI"); ax.set_ylim(0, min(500, observed.aqi_mean.max() * 1.1))
ax.legend(); ax.set_title(f"{CITY.name} daily mean AQI")
plt.tight_layout(); plt.show()

bands = observed["aqi_mean"].map(lambda v: band_for(v).label).value_counts()
print((bands / len(observed) * 100).round(1).to_string(), "\n(% of days)")

In [ ]:
# Seasonality: the single strongest structure in the series.
monthly = observed.groupby(observed.index.month)["aqi_mean"].agg(["mean", "std", "max"])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(monthly.index, monthly["mean"], yerr=monthly["std"], capsize=3, color="#a21caf", alpha=0.8)
axes[0].set_xticks(range(1, 13)); axes[0].set_title("Mean AQI by month"); axes[0].set_xlabel("month")

pivot = observed.pivot_table(index=observed.index.month, columns=observed.index.year,
                             values="aqi_mean", aggfunc="mean")
im = axes[1].imshow(pivot, aspect="auto", cmap="YlOrRd")
axes[1].set_xticks(range(len(pivot.columns)), pivot.columns)
axes[1].set_yticks(range(len(pivot.index)), pivot.index)
axes[1].set_title("Monthly mean AQI by year"); plt.colorbar(im, ax=axes[1])
plt.tight_layout(); plt.show()

In [ ]:
# Diurnal cycle, from the hourly frame. Informs whether daily aggregation
# throws away signal we should be keeping.
if not hourly.empty and "us_aqi_epa" in hourly:
    local = hourly["us_aqi_epa"].dropna().tz_convert(CITY.timezone)
    by_hour = local.groupby(local.index.hour).agg(["mean", "std"])
    fig, ax = plt.subplots()
    ax.plot(by_hour.index, by_hour["mean"], marker="o", color="#a21caf")
    ax.fill_between(by_hour.index, by_hour["mean"] - by_hour["std"],
                    by_hour["mean"] + by_hour["std"], alpha=0.2, color="#a21caf")
    ax.set_xlabel("hour (local)"); ax.set_ylabel("AQI"); ax.set_title("Diurnal cycle")
    plt.tight_layout(); plt.show()

## 2. What drives it?

The physical mechanism is dilution: concentration is roughly emissions divided
by the volume available to disperse them, and that volume is mixing-layer depth
times wind speed. If ventilation does not show up strongly here, either the
mechanism is different in Karachi or something is wrong with the features.

In [ ]:
drivers = [c for c in ["wind_speed_10m_mean", "boundary_layer_height_mean", "ventilation_index",
                       "temperature_2m_mean", "relative_humidity_2m_mean", "precipitation_sum",
                       "surface_pressure_mean", "temp_range", "dew_point_depression"]
           if c in observed.columns]
corr = observed[["aqi_mean"] + drivers].corr()["aqi_mean"].drop("aqi_mean").sort_values()

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(corr.index, corr.values, color=np.where(corr.values < 0, "#2563eb", "#dc2626"))
ax.axvline(0, color="k", lw=0.8); ax.set_xlabel("correlation with daily mean AQI")
plt.tight_layout(); plt.show()
corr.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["wind_speed_10m_mean", "boundary_layer_height_mean", "ventilation_index"]):
    if col not in observed: continue
    s = ax.scatter(observed[col], observed["aqi_mean"], c=observed.index.month,
                   cmap="twilight", s=8, alpha=0.6)
    ax.set_xlabel(col); ax.set_ylabel("AQI")
plt.colorbar(s, ax=axes[-1], label="month")
plt.tight_layout(); plt.show()

## 3. How hard is persistence to beat?

This is the most important cell in the notebook. If the lag-1 autocorrelation is
very high, "tomorrow equals today" is already a strong forecast, and any model
we build has to clear that bar before it means anything.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(observed["aqi_mean"].dropna(), lags=40, ax=axes[0])
axes[0].set_title("Autocorrelation, daily mean AQI")

for h in (1, 2, 3):
    naive_rmse = np.sqrt(((observed["aqi_mean"].shift(-h) - observed["aqi_mean"]) ** 2).mean())
    print(f"persistence RMSE at h={h}: {naive_rmse:.1f} AQI points")

axes[1].scatter(observed["aqi_mean"], observed["aqi_mean"].shift(-1), s=6, alpha=0.4)
lims = [0, observed["aqi_mean"].max()]
axes[1].plot(lims, lims, "r--", lw=1)
axes[1].set_xlabel("AQI today"); axes[1].set_ylabel("AQI tomorrow")
axes[1].set_title("Persistence, visualised")
plt.tight_layout(); plt.show()

## 4. Data quality

In [ ]:
expected = pd.date_range(observed.index.min(), observed.index.max(), freq="D")
gaps = expected.difference(observed.index)
print(f"missing calendar days: {len(gaps)}")
if len(gaps): print("examples:", [str(d.date()) for d in gaps[:10]])

if "hours_observed" in observed:
    thin = observed[observed["hours_observed"] < 18]
    print(f"days with fewer than 18 hourly observations: {len(thin)}")

nulls = (daily.isna().mean() * 100).sort_values(ascending=False)
print("\nmost incomplete features:")
print(nulls.head(12).round(1).to_string())

## 5. Does our EPA AQI agree with the API's?

We recompute the index from concentrations rather than taking the API's hourly
`us_aqi` field, because the EPA index is defined over pollutant-specific
averaging windows. The two should be correlated but not identical, and the
difference is worth documenting in the report.

In [ ]:
if not hourly.empty and {"us_aqi", "us_aqi_epa"} <= set(hourly.columns):
    pair = hourly[["us_aqi", "us_aqi_epa"]].dropna()
    if len(pair):
        diff = pair["us_aqi_epa"] - pair["us_aqi"]
        print(f"n = {len(pair):,}   correlation = {pair.corr().iloc[0,1]:.3f}")
        print(f"mean difference = {diff.mean():+.1f}   median abs = {diff.abs().median():.1f}")
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        axes[0].scatter(pair["us_aqi"], pair["us_aqi_epa"], s=3, alpha=0.25)
        lim = [0, pair.max().max()]; axes[0].plot(lim, lim, "r--", lw=1)
        axes[0].set_xlabel("Open-Meteo us_aqi (hourly)"); axes[0].set_ylabel("our EPA AQI")
        axes[1].hist(diff, bins=60, color="#a21caf", alpha=0.8)
        axes[1].set_xlabel("ours minus theirs"); axes[1].set_title("Difference")
        plt.tight_layout(); plt.show()
else:
    print("Run the backfill first.")

## Findings

*Fill in after running. These feed directly into `docs/REPORT.md`.*

1. Seasonal amplitude:
2. Strongest meteorological driver:
3. Persistence RMSE to beat (h=1/2/3):
4. Data quality issues found:
5. Our AQI versus the API's: